In [4]:
import sys, os, time
import numpy as np
import paramiko
from numpy.fft import fftshift

# ---------------- PATHS ----------------
sys.path.append('./soft')
sys.path.append("/home/xilinx/jupyter_notebooks/bread_realOrig/Linux/libusb/example")

from top import *
from helpers import *

# ---------------- CONFIG ----------------
FFT_SIZE = 16384
K_hardware = 10000
HARDWARE_OFFSET = 4096
R = 50

total_hours = 6
save_interval = 5 * 60
total_seconds = total_hours * 3600

local_dir = "/home/xilinx/dual_nyquist_6hr"
remote_dir = "C:/Users/knirc/Desktop/dual_nyquist_6hr"

hostname = "192.168.2.9"
username = "knirc"
key_path = "/home/xilinx/.ssh/id_rsa"

# ---------------- INIT HARDWARE ----------------
soc = TopSoc('../bread2NZRev.bit', ignore_version=True, force_init_clks=True)
chain = DualChain(soc, soc['analysis'][0], soc['synthesis'][0])

acc_0 = chain.soc.axis_accumulator_0
acc_1 = chain.soc.axis_accumulator_1
Fs = chain.fs

os.makedirs(local_dir, exist_ok=True)

# ---------------- SFTP ----------------
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
print(f"Connecting to {hostname}...", flush=True)

ssh.connect(hostname, username=username, key_filename=key_path, timeout=10)
sftp = ssh.open_sftp()

def ensure_remote_dir(sftp, remote_dir):
    remote_dir = remote_dir.replace("\\","/")
    current = ""
    for part in remote_dir.split("/"):
        if not part:
            continue
        current += "/" + part
        try:
            sftp.mkdir(current)
        except IOError:
            pass

ensure_remote_dir(sftp, remote_dir)
print("SFTP ready.", flush=True)

# ---------------- CORE FUNCTION ----------------
def get_dual_zone_linear(N_avg):

    _ = chain.analysis.get_data_acc(N=int(N_avg))

    # --- Extract 128-bit ---
    s_low_0 = acc_0.buff[:-1, 0].astype(np.uint64).astype(object)
    s_high_0 = acc_0.buff[:-1, 1].astype(np.uint64).astype(object)
    data_0 = s_low_0 + (s_high_0 << 64)

    s_low_1 = acc_1.buff[:-1, 0].astype(np.uint64).astype(object)
    s_high_1 = acc_1.buff[:-1, 1].astype(np.uint64).astype(object)
    data_1 = s_low_1 + (s_high_1 << 64)

    # --- Flatten ---
    flat_0 = data_0.flatten()
    flat_1 = data_1.flatten()

    # --- Align ---
    flat_0 = np.roll(flat_0, -HARDWARE_OFFSET)
    flat_1 = np.roll(flat_1, -HARDWARE_OFFSET)

    # --- FFT shift ---
    flat_0 = fftshift(flat_0)
    flat_1 = fftshift(flat_1)

    # --- Convert to float ---
    flat_0 = np.array(flat_0, dtype=np.float64)
    flat_1 = np.array(flat_1, dtype=np.float64)

    # --- Scale to linear power ---
    scale = ((0.5 / 2**15)**2) / (FFT_SIZE**2) / N_avg
    V0 = flat_0 * scale
    V1 = flat_1 * scale

    # --- FIX: Nyquist inversion ---
    V1 = np.flip(V1)

    # --- Stitch ---
    return np.concatenate((V0, V1))


# ---------------- FREQUENCY AXIS ----------------
print("Generating frequency axis...", flush=True)

test_spec = get_dual_zone_linear(K_hardware)
N = len(test_spec)

F_0 = np.linspace(0, Fs, N//2, endpoint=False)
F_1 = np.linspace(Fs, 2*Fs, N//2, endpoint=False)
F = np.concatenate((F_0, F_1))

# ---------------- ACQUISITION ----------------
print(f"Starting {total_hours}-hour run...", flush=True)

start_time = time.time()
last_save = start_time

avg_sum = None
avg_count = 0
frame = 0

while True:

    now = time.time()
    elapsed = now - start_time

    if elapsed >= total_seconds:
        break

    inst_lin = get_dual_zone_linear(1)
    avg_lin  = get_dual_zone_linear(K_hardware)

    if avg_sum is None:
        avg_sum = np.zeros_like(avg_lin)
        avg_count = 0

    avg_sum += avg_lin
    avg_count += 1
    frame += 1

    if frame % 10 == 0:
        print(f"Frame {frame} | {elapsed/60:.1f} min", flush=True)

    # -------- SAVE --------
    if now - last_save >= save_interval:

        avg_cum = avg_sum / avg_count

        inst_dbm = 10*np.log10(inst_lin/(2*R*1e-3) + 1e-12)
        avg_dbm  = 10*np.log10(avg_cum/(2*R*1e-3) + 1e-12)

        filename = f"t{int(elapsed):07d}s.csv"
        local_path = os.path.join(local_dir, filename)
        remote_path = os.path.join(remote_dir, filename)

        data = np.column_stack([F, inst_dbm, avg_dbm])
        header = "freq_MHz,inst_dBm,avg_dBm"

        np.savetxt(local_path, data, delimiter=",", header=header)

        sftp.put(local_path, remote_path)
        os.remove(local_path)

        print(f"Saved @ {elapsed/3600:.2f} hr", flush=True)

        avg_sum = None
        avg_count = 0
        last_save = now


# ---------------- FINAL SAVE ----------------
print("Final save...", flush=True)

avg_cum = avg_sum / max(avg_count, 1)

inst_dbm = 10*np.log10(inst_lin/(2*R*1e-3) + 1e-12)
avg_dbm  = 10*np.log10(avg_cum/(2*R*1e-3) + 1e-12)

local_path = os.path.join(local_dir, "FINAL.csv")
remote_path = os.path.join(remote_dir, "FINAL.csv")

data = np.column_stack([F, inst_dbm, avg_dbm])
header = "freq_MHz,inst_dBm,avg_dBm"

np.savetxt(local_path, data, delimiter=",", header=header)
sftp.put(local_path, remote_path)
os.remove(local_path)

print("Done.", flush=True)

sftp.close()
ssh.close()

resetting clocks: 245.76 491.52
Connecting to 192.168.2.9...
SFTP ready.
Generating frequency axis...
Starting 6-hour run...
Frame 10 | 0.3 min
Frame 20 | 0.7 min
Frame 30 | 1.0 min
Frame 40 | 1.4 min
Frame 50 | 1.7 min
Frame 60 | 2.1 min
Frame 70 | 2.4 min
Frame 80 | 2.8 min
Frame 90 | 3.1 min
Frame 100 | 3.5 min
Frame 110 | 3.8 min
Frame 120 | 4.2 min
Frame 130 | 4.5 min
Frame 140 | 4.9 min
Saved @ 0.08 hr
Frame 150 | 5.4 min
Frame 160 | 5.8 min
Frame 170 | 6.1 min
Frame 180 | 6.5 min
Frame 190 | 6.8 min
Frame 200 | 7.2 min
Frame 210 | 7.5 min
Frame 220 | 7.9 min
Frame 230 | 8.2 min
Frame 240 | 8.6 min
Frame 250 | 8.9 min
Frame 260 | 9.3 min
Frame 270 | 9.6 min
Frame 280 | 10.0 min
Saved @ 0.17 hr
Frame 290 | 10.5 min
Frame 300 | 10.8 min
Frame 310 | 11.2 min
Frame 320 | 11.5 min
Frame 330 | 11.9 min
Frame 340 | 12.2 min
Frame 350 | 12.6 min
Frame 360 | 12.9 min
Frame 370 | 13.3 min
Frame 380 | 13.6 min
Frame 390 | 14.0 min
Frame 400 | 14.3 min
Frame 410 | 14.7 min
Frame 420 | 15.1 m

Frame 3530 | 128.2 min
Frame 3540 | 128.5 min
Frame 3550 | 128.9 min
Frame 3560 | 129.2 min
Frame 3570 | 129.6 min
Frame 3580 | 129.9 min
Frame 3590 | 130.3 min
Saved @ 2.17 hr
Frame 3600 | 130.8 min
Frame 3610 | 131.1 min
Frame 3620 | 131.5 min
Frame 3630 | 131.8 min
Frame 3640 | 132.2 min
Frame 3650 | 132.5 min
Frame 3660 | 132.9 min
Frame 3670 | 133.3 min
Frame 3680 | 133.6 min
Frame 3690 | 134.0 min
Frame 3700 | 134.3 min
Frame 3710 | 134.7 min
Frame 3720 | 135.0 min
Frame 3730 | 135.4 min
Saved @ 2.26 hr
Frame 3780 | 137.3 min
Frame 3790 | 137.6 min
Frame 3800 | 138.0 min
Frame 3810 | 138.4 min
Frame 3820 | 138.7 min
Frame 3830 | 139.1 min
Frame 3840 | 139.4 min
Frame 3850 | 139.8 min
Frame 3860 | 140.1 min
Frame 3870 | 140.5 min
Saved @ 2.34 hr
Frame 3880 | 141.0 min
Frame 3890 | 141.3 min
Frame 3900 | 141.7 min
Frame 3910 | 142.0 min
Frame 3920 | 142.4 min
Frame 3930 | 142.7 min
Frame 3940 | 143.1 min
Frame 3950 | 143.4 min
Frame 3960 | 143.8 min
Frame 3970 | 144.2 min
Frame 398

Frame 6960 | 253.2 min
Frame 6970 | 253.5 min
Frame 6980 | 253.9 min
Frame 6990 | 254.2 min
Frame 7000 | 254.6 min
Frame 7010 | 254.9 min
Frame 7020 | 255.3 min
Frame 7030 | 255.6 min
Frame 7040 | 256.0 min
Saved @ 4.27 hr
Frame 7050 | 256.5 min
Frame 7060 | 256.8 min
Frame 7070 | 257.2 min
Frame 7080 | 257.6 min
Frame 7090 | 257.9 min
Frame 7100 | 258.3 min
Frame 7110 | 258.6 min
Frame 7120 | 259.0 min
Frame 7130 | 259.3 min
Frame 7140 | 259.7 min
Frame 7150 | 260.0 min
Frame 7160 | 260.4 min
Frame 7170 | 260.7 min
Frame 7180 | 261.1 min
Saved @ 4.35 hr
Frame 7190 | 261.6 min
Frame 7200 | 261.9 min
Frame 7210 | 262.3 min
Frame 7220 | 262.7 min
Frame 7230 | 263.0 min
Frame 7240 | 263.4 min
Frame 7250 | 263.7 min
Frame 7260 | 264.1 min
Frame 7270 | 264.4 min
Frame 7280 | 264.8 min
Frame 7290 | 265.1 min
Frame 7300 | 265.5 min
Frame 7310 | 265.8 min
Saved @ 4.44 hr
Frame 7320 | 266.3 min
Frame 7330 | 266.7 min
Frame 7340 | 267.1 min
Frame 7350 | 267.4 min
Frame 7360 | 267.8 min
Frame 737

In [2]:
# must run this between averaging runs and changing firmwmares before starting another data-taking run
soc = TopSoc('../bread2NZRev.bit', ignore_version=True, force_init_clks=True)

resetting clocks: 245.76 491.52
